## EDA Set-up cleaning Metaclean dataset

# 1 import and data loading

Import the requiered libaries and load the orginical excel dataset

In [17]:
import pandas as pd
import re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)


In [48]:
#Load the excel metaclean dataset files.
#The code opens the excel file and loads all the movie information into Python.

metaclean = pd.read_excel(
    r'/Users/tenzinphuntsok/Desktop/Metacritic dataset/metaClean43Brightspace.xlsx'
)

# Display confirmation
print("Dataset successfully loaded!")

# Display number of rows and columns
print("Rows and columns:", metaclean.shape)

# Display first 5 rows
metaclean.head()

Dataset successfully loaded!
Rows and columns: (11364, 13)


,url,title,studio,rating,runtime,cast,director,genre,summary,awards,metascore,userscore,RelDate
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,| Not Rated,83.0,NaN,Lynn Hershman-Leeson,Documentary,NaN,NaN,70,NaN,2011-06-01
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,| PG-13,104.0,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","Waking up from a car accident, a young woman (...","#18MostDiscussedMovieof2016 , #1MostSharedMovi...",76,7.7,2016-03-11
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,| R,82.0,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"Drama,Comedy,Romance",While researching a role as a supermarket mana...,NaN,54,5.8,2006-12-01
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,| R,100.0,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"Drama,Comedy,Romance",NaN,NaN,61,6.9,2012-09-14
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,| Not Rated,91.0,NaN,Cameron Cairnes,"Horror,Comedy",Reg and Lindsay run an organic fertilizer busi...,NaN,63,7.5,2013-06-28


* The ouput indicates that the dataset contains 11,364 movie records and 13 different variables. 
* Each row represents a movie record, while each column provides information such as title, studio, genre, runtime and scores about the movie.


## 2 Data inspection

This section is to understan the dataset before making any changes. By inspecting the columns, data types, misssing values and duplicate records.

In [19]:
# Display the column names and their data types
print("=== Dataset columns overview ===")

print(metaclean.dtypes)

=== Dataset columns overview ===
url                  object
title                object
studio               object
rating               object
runtime             float64
cast                 object
director             object
genre                object
summary              object
awards               object
metascore             int64
userscore           float64
RelDate      datetime64[ns]
dtype: object


The output indicates three main types of information:
- Text: movie titles, studios, directors and genres.
- Numerical: runtime, Metascore and User Score.
- Dates: movie release dates.

However, the release date is already recognised as a date, and the numerical scores are recognised as numbers. This is useful because Python can work with these variables correctly during later analysis.

In [ ]:
# Check the data types of each column 
metaclean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11364 entries, 0 to 11363
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   url        11364 non-null  object        
 1   title      11364 non-null  object        
 2   studio     11014 non-null  object        
 3   rating     10297 non-null  object        
 4   runtime    11109 non-null  float64       
 5   cast       7662 non-null   object        
 6   director   11350 non-null  object        
 7   genre      11344 non-null  object        
 8   summary    5467 non-null   object        
 9   awards     4387 non-null   object        
 10  metascore  11364 non-null  int64         
 11  userscore  9259 non-null   float64       
 12  RelDate    11364 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), object(9)
memory usage: 1.1+ MB


The results provides a more detailed overview of the dataset, including how many records contain information and how many are empty in each column.

Highlighted output:
- Total records: 11,364.
- Total columns: 13.
- Data types: 9 text, 2 decimal, 1 whole number and 1 date.

So, the output confirms the structure of the dataset and helps identify which variables may require attention.

In [ ]:
#Summary statistics for numerical columns
#This calculates basic statistics for your numerical columns. It helps us understand the typical values, averages and ranges in the dataset.
metaclean.describe()

,runtime,metascore,userscore,RelDate
count,11109.000000,11364.000000,9259.000000,11364
mean,102.686110,58.392555,6.542672,2011-11-05 06:50:56.388595712
min,21.000000,1.000000,0.200000,2000-01-01 00:00:00
25%,91.000000,47.000000,5.800000,2006-07-14 00:00:00
50%,100.000000,60.000000,6.700000,2012-07-20 00:00:00
75%,111.000000,71.000000,7.500000,2017-01-27 00:00:00
max,808.000000,100.000000,9.700000,2021-12-29 00:00:00
std,19.952258,16.898457,1.374956,NaN


The output indicates:
1. Movie runtimes average approximately 103 minutes. Most movies are around 100 minutes long.
2. The average Metascore is approximately 58, suggesting that the typical critic score in this dataset is around the middle of the 0–100 scale.
3. The average User Score is approximately 6.5 out of 10, indicating a moderately positive average audience score.
4. The dataset contains movie releases from 2000 to 2021.

The result is very useful because it gives an initial understanding of movie characteristics and performance scores.

In [21]:
#Count the missing values in each column of the dataset
print("=== Missing values ====")

print(metaclean.isnull().sum())

=== Missing values ====
url             0
title           0
studio        350
rating       1067
runtime       255
cast         3702
director       14
genre          20
summary      5897
awards       6977
metascore       0
userscore    2105
RelDate         0
dtype: int64


The output shows that the dataset has different levels of missing information. 

- Awards has the most missing values: 6,977 (61.4%).
- Summary has 5,897 missing values (51.9%).
- Cast has 3,702 missing values (32.6%).
- User Score has 2,105 missing values (18.5%).

On the other hand, URL, title, Metascore and release date have no missing values.

In [22]:
# Check for duplicate rows in the dataset
print("=== Duplicate rows ===")

print(metaclean.duplicated().sum())

=== Duplicate rows ===
0


The output indicates that every row has at least one difference compared with the other rows.

# 3 Basic data cleaning

Make small formatting corrections to improve consistancy without changing the original movie information.

In [43]:
#Creating a copy of the original dataset to avoid any accidental changes to the original data.
#This way we can always refer back to the original dataset if needed.

metaclean_cleaned = metaclean.copy()

print("Orginal dataset copied successfully!")

Orginal dataset copied successfully!


In [ ]:
# Removes spaces from the text values inside the columns (before and after the column names)
# This helps make the information more consistent and easier to work with.

# Identify text columns

text_columns = metaclean_cleaned.select_dtypes(
    include='object'
).columns

# Remove leading and trailing spaces only

for column in text_columns:

    metaclean_cleaned[column] = metaclean_cleaned[column].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

This part identifies all text columns and removes unnecessary spaces at the beginning and end of each text entry.

For example:
- " Drama " becomes "Drama".
- " Paramount Pictures " becomes "Paramount Pictures".

In [45]:
#Stanardize the text to lowercase

lowercase_columns = [
    'genre',
    'summary',
    'award',
    'rating'
]

#Convert selected columns to lowercase
for column in lowercase_columns:

    if column in metaclean_cleaned.columns:

        metaclean_cleaned[column] = metaclean_cleaned [column].apply(
            lambda x: x.lower() if isinstance(x, str) else x
        )

This next step changes selected text values into lowercase so that the text has a consistent format.

For example:
- Documentary becomes documentary.
- Action, Drama becomes action, drama.

In [ ]:
# Checks whether the original and cleaned datasets still contain the same number of rows and columns.
# Compare original and cleaned dataset dimensions

print("Original dataset:", metaclean.shape)

print("Cleaned dataset:", metaclean_cleaned.shape)

Original dataset: (11364, 13)
Cleaned dataset: (11364, 13)


In [ ]:
# Original dataset
print("=== Original dataset ===") 
display(metaclean.head())

# Cleaned dataset
print("=== Cleaned dataset ===")
display(metaclean_cleaned.head())

=== Original dataset ===


,url,title,studio,rating,runtime,cast,director,genre,summary,awards,metascore,userscore,RelDate
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,| Not Rated,83.0,NaN,Lynn Hershman-Leeson,Documentary,NaN,NaN,70,NaN,2011-06-01
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,| PG-13,104.0,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","Waking up from a car accident, a young woman (...","#18MostDiscussedMovieof2016 , #1MostSharedMovi...",76,7.7,2016-03-11
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,| R,82.0,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"Drama,Comedy,Romance",While researching a role as a supermarket mana...,NaN,54,5.8,2006-12-01
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,| R,100.0,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"Drama,Comedy,Romance",NaN,NaN,61,6.9,2012-09-14
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,| Not Rated,91.0,NaN,Cameron Cairnes,"Horror,Comedy",Reg and Lindsay run an organic fertilizer busi...,NaN,63,7.5,2013-06-28


=== Cleaned dataset ===


,url,title,studio,rating,runtime,cast,director,genre,summary,awards,metascore,userscore,RelDate
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,| not rated,83.0,NaN,Lynn Hershman-Leeson,documentary,NaN,NaN,70,NaN,2011-06-01
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,| pg-13,104.0,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"action,sci-fi,drama,mystery,thriller,horror","waking up from a car accident, a young woman (...","#18MostDiscussedMovieof2016 , #1MostSharedMovi...",76,7.7,2016-03-11
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,| r,82.0,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"drama,comedy,romance",while researching a role as a supermarket mana...,NaN,54,5.8,2006-12-01
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,| r,100.0,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"drama,comedy,romance",NaN,NaN,61,6.9,2012-09-14
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,| not rated,91.0,NaN,Cameron Cairnes,"horror,comedy",reg and lindsay run an organic fertilizer busi...,NaN,63,7.5,2013-06-28


Here it is the table of the original and cleaned data set. It displays the first five movies before and after cleaning so you can visually check what has changed.

What the actual output indicates:
- Movie titles remain unchanged.
- Studio names remain unchanged.
- Ratings have been converted to lowercase.
- Director names remain unchanged.
- Genre names are now lowercase.
- Summaries have been converted to lowercase.
- Missing information remains missing.
- Numerical scores and release dates remain unchanged.

In [49]:
# Save the cleaned dataset in my file on laptop.

metaclean_cleaned.to_excel(
    r'/Users/tenzinphuntsok/Desktop/Metacritic dataset/metaClean43Brightspace_cleaned.xlsx',
    index=False
)

print("Cleaned dataset successfully saved!")

Cleaned dataset successfully saved!
